<a href="https://colab.research.google.com/github/adikatre/Asymmetric-Cross-Modal-Attention/blob/main/notebooks/train_evaluate_visualize_colab_frozen_a100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Train, Evaluate & Visualize — A100-optimized variant

> **A100 notes.** This is the A100-tuned copy of the frozen-encoder notebook. Differences vs. the original:
> - Mixed precision uses **bfloat16** autocast on Ampere+ (A100) instead of fp16; the `GradScaler` is disabled under bf16 (no loss scaling needed). On a non-Ampere GPU it falls back to fp16 automatically.
> - Uses the modern `torch.amp` API (the deprecated `torch.cuda.amp` import is gone).
> - **TF32** matmul/conv and `cudnn.benchmark` remain enabled (already present in the original).
> - **Batch size** raised for A100 headroom: 256 (frozen) / 64 (end-to-end). Learning rate is unchanged. If you hit OOM, lower `BATCH_SIZE` in the config cell.


## Load and Unzip VQA Dataset

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/sample_data/

Mounted at /content/drive


In [2]:
# make data dirs
!mkdir -p /content/data/
!mkdir -p /content/data/answers
!mkdir -p /content/data/images
!mkdir -p /content/data/questions

#copy over zips (https://drive.google.com/drive/folders/1VJ1xNxo_dAGJ4ZcpaFQBo-wpdIChZkpx?usp=sharing) from drive into here
!cp -r /content/drive/MyDrive/VQA/ /content/data/zip/

In [3]:
# extract annotations (labels)
!unzip -q /content/data/zip/v2_Annotations_Train_mscoco.zip -d /content/data/answers/
!unzip -q /content/data/zip/v2_Annotations_Val_mscoco.zip -d /content/data/answers/

!rm -rf /content/data/zip/v2_Annotations_Train_mscoco.zip
!rm -rf /content/data/zip/v2_Annotations_Val_mscoco.zip

In [4]:
# extract testing data
!unzip -q /content/data/zip/v2_Questions_Test_mscoco.zip -d /content/data/questions/
!unzip -q /content/data/zip/v2_Questions_Train_mscoco.zip -d /content/data/questions/
!unzip -q /content/data/zip/v2_Questions_Val_mscoco.zip -d /content/data/questions/

!rm -rf /content/data/zip/v2_Questions_Test_mscoco.zip
!rm -rf /content/data/zip/v2_Questions_Train_mscoco.zip
!rm -rf /content/data/zip/v2_Questions_Val_mscoco.zip

In [5]:
# copy over past results for resume function
!mkdir -p /content/results
!cp -r /content/drive/MyDrive/unfrozen_results_ViT_CLIP_RoBERTa/results/* /content/results/

In [6]:
# Optional: reuse a prebuilt 336px HDF5 from Drive if it exists.
# If this file is absent, the preprocessing cell below will regenerate it
# from raw train2014/val2014 JPEGs.
!mkdir -p /content/data/
!cp /content/drive/MyDrive/VQA_cache/vqa_images_336.h5 /content/data/vqa_images_336.h5


# 2 — Train, Evaluate & Visualize

Complete experiment in one notebook:

1. **Data** — Load VQA v2.0 dataset
2. **Models** — Define encoders, attention blocks, and full VQA models
3. **Training** — Train symmetric baseline and asymmetric model
4. **Evaluation** — Compare metrics (Top-1, Top-5 accuracy)
5. **Visualization** — Training curves, attention heatmaps, qualitative examples

**Run all cells in order.** Edit the configuration cell below to change hyperparameters.

In [7]:
!pip install -q torch torchvision transformers matplotlib tqdm Pillow h5py


In [8]:
import json
import math
import random
import re
import string
import time
from collections import Counter
from contextlib import nullcontext
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm import tqdm
from transformers import (
    CLIPVisionModel,
    RobertaModel,
    RobertaTokenizer,
    get_cosine_schedule_with_warmup,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Throughput flags (Ampere/A100): TF32 matmuls + autotuned cuDNN kernels.
# Free speedup, no measurable accuracy impact.
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# Prefer bfloat16 autocast on Ampere+ (A100): fp32 exponent range, no loss scaling.
# Fall back to fp16 on older GPUs so the notebook still runs there.
AMP_DTYPE = (
    torch.bfloat16
    if (device.type == "cuda" and torch.cuda.is_bf16_supported())
    else torch.float16
)


Device: cuda


## Configuration

Edit these variables to change the experiment.

In [ ]:
# Paths
DATA_DIR        = Path("/content/data")
CHECKPOINT_DIR = Path("/content/results/checkpoints")
METRICS_DIR    = Path("/content/results/metrics")
FIGURES_DIR    = Path("/content/results/figures")

# Model
NUM_ANSWERS      = 3000
EMBED_DIM        = 512
NUM_HEADS        = 8
DROPOUT          = 0.2
FREEZE_ENCODERS  = False  # True = offline features; False = end-to-end partial fine-tuning

# CLIP ViT-L/14@336 image encoder: 24x24 patch grid + CLS = 577 tokens.
IMAGE_ENCODER_NAME   = "openai/clip-vit-large-patch14-336"
IMAGE_SIZE           = 336
IMAGE_PATCH_SIZE     = 14
IMAGE_SEQ_LEN        = (IMAGE_SIZE // IMAGE_PATCH_SIZE) ** 2 + 1
UNFREEZE_VISION_LAYERS = 4

# Run tagging prevents accidental resume from older ViT-B/16, top-1000 checkpoints.
RUN_TAG = "clipL336_vqaScore_top3000_s42"

# Data
MAX_QUESTION_LEN = 20
MAX_SAMPLES      = None    # set to 1000 for a quick dev run, and None for a complete run

# Training
if FREEZE_ENCODERS:
    BATCH_SIZE       = 256    # A100: raised from 128; lower if OOM; LR unchanged
    LEARNING_RATE    = 2e-4
    ENCODER_LR       = 0.0
    WARMUP_EPOCHS    = 0
    GRAD_ACCUM_STEPS = 1
else:
    # GPU was starved at batch 8 (~5.6/22.5 GB). A real batch of 32 keeps the same
    # effective batch (no accumulation) but feeds the GPU properly -> much faster,
    # accuracy-neutral (LayerNorm model, so batch-size-independent).
    BATCH_SIZE       = 64     # A100: raised from 32; lower if OOM; LR unchanged
    LEARNING_RATE    = 2e-4   # fusion + classifier + projection LR
    ENCODER_LR       = 2e-6   # unfrozen CLIP/RoBERTa encoder LR
    WARMUP_EPOCHS    = 1
    GRAD_ACCUM_STEPS = 1      # effective batch stays 32, so LR is unchanged

WEIGHT_DECAY        = 1e-2
MAX_GRAD_NORM       = 1.0
EPOCHS              = 13
NUM_WORKERS         = 8       # 336px decode + transform is CPU-bound; was 4
SEED                = 7
USE_AMP             = True
RUN_SMOKE_TEST      = False
SMOKE_TEST_STEPS    = 200
REPORT_TOP1000_COMPAT = True

for d in [CHECKPOINT_DIR, METRICS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)


---
## 1. Data Loading

In [10]:
# CLIP preprocessing constants for openai/clip-vit-large-patch14-336.
CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD  = [0.26862954, 0.26130258, 0.27577711]

NUMBER_MAP = {
    "zero": "0", "one": "1", "two": "2", "three": "3", "four": "4",
    "five": "5", "six": "6", "seven": "7", "eight": "8", "nine": "9",
    "ten": "10", "eleven": "11", "twelve": "12", "thirteen": "13",
    "fourteen": "14", "fifteen": "15", "sixteen": "16", "seventeen": "17",
    "eighteen": "18", "nineteen": "19", "twenty": "20",
}
ARTICLES = {"a", "an", "the"}
CONTRACTIONS = {
    "dont": "don't", "doesnt": "doesn't", "didnt": "didn't",
    "isnt": "isn't", "arent": "aren't", "wasnt": "wasn't", "werent": "weren't",
    "cant": "can't", "couldnt": "couldn't", "wouldnt": "wouldn't", "shouldnt": "shouldn't",
    "hasnt": "hasn't", "havent": "haven't", "wont": "won't", "whats": "what's",
}
PUNCT_TO_SPACE = "".join(ch for ch in string.punctuation if ch not in {"'", ":", ".", ","})
PUNCT_RE = re.compile(f"[{re.escape(PUNCT_TO_SPACE)}]")


def normalize_answer(answer):
    """VQA-style answer normalization used for vocab, targets, eval, and export."""
    text = str(answer).lower().strip()
    text = text.replace("\n", " ").replace("\t", " ")
    text = re.sub(r"(?<=\d),(?=\d)", "", text)       # 100,978 -> 100978
    text = text.replace(",", " ")
    text = re.sub(r"(?<!\d)\.(?!\d)", " ", text)     # keep decimal periods
    text = PUNCT_RE.sub(" ", text)

    tokens = []
    for token in text.split():
        token = NUMBER_MAP.get(token, token)
        token = CONTRACTIONS.get(token, token)
        if token not in ARTICLES:
            tokens.append(token)
    return " ".join(tokens)


def build_answer_vocab(annotations_file, top_k=3000):
    """Build answer vocabulary from the top_k most frequent normalized answers."""
    with open(annotations_file) as f:
        annotations = json.load(f)["annotations"]

    counter = Counter()
    for ann in annotations:
        ans = normalize_answer(ann["multiple_choice_answer"])
        if ans:
            counter[ans] += 1

    most_common = [ans for ans, _ in counter.most_common(top_k)]
    answer_to_idx = {ans: idx for idx, ans in enumerate(most_common)}
    idx_to_answer = {idx: ans for ans, idx in answer_to_idx.items()}
    return answer_to_idx, idx_to_answer


def get_image_transform(split="train"):
    """CLIP-normalized transform for ViT-L/14@336."""
    interpolation = transforms.InterpolationMode.BICUBIC
    if split == "train":
        return transforms.Compose([
            transforms.RandomResizedCrop(
                IMAGE_SIZE, scale=(0.85, 1.0), ratio=(0.9, 1.1),
                interpolation=interpolation),
            transforms.ToTensor(),
            transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
        ])
    return transforms.Compose([
        transforms.Resize(IMAGE_SIZE, interpolation=interpolation),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
    ])


def get_h5_preprocess_transform():
    """Resize/crop raw JPEGs to IMAGE_SIZE for versioned HDF5 storage."""
    interpolation = transforms.InterpolationMode.BICUBIC
    return transforms.Compose([
        transforms.Resize(IMAGE_SIZE, interpolation=interpolation),
        transforms.CenterCrop(IMAGE_SIZE),
    ])


## Storage as one .h5 file

Images are stored in a versioned HDF5 file at `IMAGE_SIZE x IMAGE_SIZE` so the 336px CLIP run never reuses the older 256px cache.


In [11]:
# --- HDF5 Preprocessing: convert raw JPEGs into a single contiguous file ---
h5_path = DATA_DIR / f"vqa_images_{IMAGE_SIZE}.h5"

if h5_path.exists():
    with h5py.File(h5_path, "r") as h5f:
        stored_shape = tuple(h5f["images"].shape[1:3])
    if stored_shape != (IMAGE_SIZE, IMAGE_SIZE):
        raise ValueError(
            f"{h5_path} has image shape {stored_shape}, expected {(IMAGE_SIZE, IMAGE_SIZE)}. "
            "Delete/regenerate it instead of reusing an incompatible cache.")
    print(f"HDF5 file already exists and matches {IMAGE_SIZE}px: {h5_path}")
else:
    preprocess = get_h5_preprocess_transform()
    image_dirs = [DATA_DIR / "images" / "train2014", DATA_DIR / "images" / "val2014"]

    # Collect all image paths
    all_paths = []
    for d in image_dirs:
        if d.exists():
            all_paths.extend(sorted(d.glob("*.jpg")))
    print(f"Found {len(all_paths):,} images to preprocess at {IMAGE_SIZE}px")

    with h5py.File(h5_path, "w") as h5f:
        imgs_ds = h5f.create_dataset(
            "images", shape=(len(all_paths), IMAGE_SIZE, IMAGE_SIZE, 3),
            dtype=np.uint8, chunks=(1, IMAGE_SIZE, IMAGE_SIZE, 3))
        ids_ds = h5f.create_dataset(
            "image_ids", shape=(len(all_paths),), dtype=np.int64)

        for i, path in enumerate(tqdm(all_paths, desc="Preprocessing images")):
            image_id = int(path.stem.split("_")[-1])
            img = Image.open(path).convert("RGB")
            img = preprocess(img)
            imgs_ds[i] = np.array(img)
            ids_ds[i] = image_id

    print(f"Saved {len(all_paths):,} images to {h5_path}")


HDF5 file already exists and matches 336px: /content/data/vqa_images_336.h5


In [12]:
class VQADataset(Dataset):
    """PyTorch dataset for VQA v2.0 with HDF5 image loading and VQA-score targets.

    Returns (image_tensor, input_ids, attention_mask, answer_target) tuples.
    answer_target is a multi-label score vector where each answer receives
    min(num_matching_annotators / 3, 1), matching the standard VQA metric.
    """

    def __init__(self, questions_file, annotations_file, h5_path,
                 answer_to_idx=None, top_k_answers=3000,
                 max_question_len=20, transform=None, max_samples=None):
        self.h5_path = Path(h5_path)
        self.max_question_len = max_question_len
        self.transform = transform or get_image_transform("val")
        self.tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
        self._h5f = None  # lazy-opened per DataLoader worker

        if answer_to_idx is None:
            self.answer_to_idx, self.idx_to_answer = build_answer_vocab(
                annotations_file, top_k_answers)
        else:
            self.answer_to_idx = answer_to_idx
            self.idx_to_answer = {v: k for k, v in answer_to_idx.items()}

        # Build image_id -> HDF5 row index mapping
        with h5py.File(self.h5_path, "r") as f:
            image_ids = f["image_ids"][:]
        self.id_to_row = {int(iid): i for i, iid in enumerate(image_ids)}

        with open(questions_file) as f:
            questions_data = json.load(f)["questions"]
        with open(annotations_file) as f:
            annotations_data = json.load(f)["annotations"]

        ann_by_qid = {ann["question_id"]: ann for ann in annotations_data}
        num_answers = len(self.answer_to_idx)

        self.samples = []
        for q in questions_data:
            ann = ann_by_qid.get(q["question_id"])
            if ann is None:
                continue

            target = torch.zeros(num_answers, dtype=torch.float)
            answer_counts = Counter(normalize_answer(a["answer"]) for a in ann["answers"])
            for ans_text, count in answer_counts.items():
                if ans_text in self.answer_to_idx:
                    target[self.answer_to_idx[ans_text]] = min(count / 3.0, 1.0)

            # Skip questions where no annotator answer falls within the top-k vocabulary
            if target.sum() == 0:
                continue

            self.samples.append({
                "question": q["question"],
                "image_id": q["image_id"],
                "answer_target": target,
            })
            if max_samples is not None and len(self.samples) >= max_samples:
                break

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # Lazy HDF5 open (per-worker for multiprocessing safety)
        if self._h5f is None:
            self._h5f = h5py.File(self.h5_path, "r")

        row = self.id_to_row[sample["image_id"]]
        img_array = self._h5f["images"][row]  # (IMAGE_SIZE, IMAGE_SIZE, 3) uint8
        image = Image.fromarray(img_array)
        image = self.transform(image)

        encoding = self.tokenizer(
            sample["question"],
            max_length=self.max_question_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        return image, input_ids, attention_mask, sample["answer_target"]


In [13]:
# Build answer vocab from training annotations
train_ann = DATA_DIR / "answers" / "v2_mscoco_train2014_annotations.json"
answer_to_idx, idx_to_answer = build_answer_vocab(train_ann, NUM_ANSWERS)
print(f"Answer vocab: {len(answer_to_idx)} normalized classes")

# Create datasets (loading images from versioned HDF5)
train_ds = VQADataset(
    questions_file=DATA_DIR / "questions" / "v2_OpenEnded_mscoco_train2014_questions.json",
    annotations_file=train_ann,
    h5_path=h5_path,
    answer_to_idx=answer_to_idx,
    max_question_len=MAX_QUESTION_LEN,
    transform=get_image_transform("train"),
    max_samples=MAX_SAMPLES,
)
val_ds = VQADataset(
    questions_file=DATA_DIR / "questions" / "v2_OpenEnded_mscoco_val2014_questions.json",
    annotations_file=DATA_DIR / "answers" / "v2_mscoco_val2014_annotations.json",
    h5_path=h5_path,
    answer_to_idx=answer_to_idx,
    max_question_len=MAX_QUESTION_LEN,
    transform=get_image_transform("val"),
    max_samples=MAX_SAMPLES,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_ds):,} samples ({len(train_loader)} batches)")
print(f"Val:   {len(val_ds):,} samples ({len(val_loader)} batches)")

# Save both the latest vocab path and a versioned copy for this run.
with open(CHECKPOINT_DIR / "answer_vocab.json", "w") as f:
    json.dump(answer_to_idx, f, indent=2)
with open(CHECKPOINT_DIR / f"{RUN_TAG}_answer_vocab.json", "w") as f:
    json.dump(answer_to_idx, f, indent=2)


Answer vocab: 3000 normalized classes


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Train: 435,080 samples (13597 batches)
Val:   209,782 samples (6556 batches)


---
## 2. Model Definitions

### Encoders

The upgraded run keeps the fusion architecture intact but swaps the image tower to **CLIP ViT-L/14@336**:

- **ImageEncoder** — CLIP ViT-L/14@336: produces 577 tokens (576 spatial + 1 CLS), projected from 1024 -> `EMBED_DIM`. Most CLIP layers are frozen; only the last few vision layers are fine-tuned.
- **TextEncoder** — RoBERTa-base: produces per-token embeddings, projected from 768 -> `EMBED_DIM`.


In [14]:
class ImageEncoder(nn.Module):
    """CLIP ViT-L/14@336 image encoder. Output: (B, 577, embed_dim)."""

    def __init__(self, embed_dim=512, freeze=True,
                 model_name=IMAGE_ENCODER_NAME,
                 unfreeze_last_n=UNFREEZE_VISION_LAYERS):
        super().__init__()
        self.clip = CLIPVisionModel.from_pretrained(model_name)
        self.hidden_dim = self.clip.config.hidden_size
        self.patch_size = self.clip.config.patch_size
        self.image_size = self.clip.config.image_size
        self.projection = nn.Linear(self.hidden_dim, embed_dim)

        # A100-safe partial fine-tuning: keep most of CLIP-L frozen and adapt
        # only the final vision blocks plus the projection layer.
        for param in self.clip.parameters():
            param.requires_grad = False

        if not freeze and unfreeze_last_n > 0:
            # transformers >=5 flattens CLIPVisionModel (no .vision_model wrapper);
            # older versions nest the transformer under .vision_model.
            vision = getattr(self.clip, "vision_model", self.clip)
            for layer in vision.encoder.layers[-unfreeze_last_n:]:
                for param in layer.parameters():
                    param.requires_grad = True
            if hasattr(vision, "post_layernorm"):
                for param in vision.post_layernorm.parameters():
                    param.requires_grad = True

    def forward(self, images):
        outputs = self.clip(pixel_values=images)
        return self.projection(outputs.last_hidden_state)  # (B, 577, embed_dim)


class TextEncoder(nn.Module):
    """RoBERTa-base text encoder. Output: (B, seq_len, embed_dim)."""

    ROBERTA_HIDDEN_DIM = 768

    def __init__(self, embed_dim=512, freeze=True):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained("roberta-base")
        self.projection = nn.Linear(self.ROBERTA_HIDDEN_DIM, embed_dim)

        if freeze:
            for param in self.roberta.parameters():
                param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        return self.projection(outputs.last_hidden_state)  # (B, seq_len, embed_dim)


### Cross-Attention Blocks

The core research contribution. Cross-attention lets one modality "ask questions" of the other:

- **Asymmetric**: Two **independent** blocks with separate weights — one for image→text, one for text→image
- **Symmetric** (baseline): A **single shared** block used in both directions — cannot learn directional patterns

In [15]:
class CrossAttentionBlock(nn.Module):
    """Cross-attention: queries from one modality attend to keys/values from another.
    Includes LayerNorm, residual connections, and a feed-forward network."""

    def __init__(self, embed_dim, num_heads=8, dropout=0.1):
        super().__init__()
        # Pre-norm design: LayerNorm is applied BEFORE attention (not after).
        # This improves training stability for deep networks compared to post-norm.
        self.norm_q = nn.LayerNorm(embed_dim)
        self.norm_kv = nn.LayerNorm(embed_dim)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm_ff = nn.LayerNorm(embed_dim)
        # Standard Transformer FFN: expand 4x with GELU activation, then project back
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, query, key_value, key_padding_mask=None):
        # Pre-norm cross-attention: normalize inputs, attend, then ADD back the
        # original query (residual). This preserves the raw query signal while
        # enriching it with cross-modal context from key_value.
        q = self.norm_q(query)
        kv = self.norm_kv(key_value)
        attended, attn_weights = self.cross_attn(
            q, kv, kv, key_padding_mask=key_padding_mask,
            need_weights=True, average_attn_weights=True)  # weights saved for visualization
        query = query + attended  # residual connection
        # Pre-norm feed-forward + residual (same pattern)
        query = query + self.ff(self.norm_ff(query))
        return query, attn_weights


class AsymmetricCrossModalFusion(nn.Module):
    """Two SEPARATE cross-attention blocks with DIFFERENT learned weights,
    executed SEQUENTIALLY — this is the core research contribution.

    Step 1: image attends to text  -> img_attended  (visually-grounded image features)
    Step 2: text attends to img_attended (NOT raw image!) -> txt_attended

    The asymmetry is twofold:
      1. Each direction has its own independently learned weights
      2. Information flows sequentially: text benefits from the already-grounded
         image features, creating a deeper cross-modal interaction than parallel fusion.
    """

    def __init__(self, embed_dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.cross_attn_img_to_txt = CrossAttentionBlock(embed_dim, num_heads, dropout)
        self.cross_attn_txt_to_img = CrossAttentionBlock(embed_dim, num_heads, dropout)

    def forward(self, image_features, text_features, text_padding_mask=None):
        # Step 1: image queries attend to text keys/values
        img_attended, attn_i2t = self.cross_attn_img_to_txt(
            query=image_features, key_value=text_features,
            key_padding_mask=text_padding_mask)
        # Step 2: text queries attend to the ATTENDED image features (output of Step 1),
        # NOT the raw image features. This lets the text representation leverage the
        # image-text grounding that was already learned in Step 1.
        txt_attended, attn_t2i = self.cross_attn_txt_to_img(
            query=text_features, key_value=img_attended)
        return img_attended, txt_attended, attn_i2t, attn_t2i


class SymmetricCrossModalFusion(nn.Module):
    """ONE SHARED cross-attention block used in both directions (baseline).
    Both directions share the same weights and operate on RAW encoder outputs
    independently — no sequential information flow between them."""

    def __init__(self, embed_dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.shared_cross_attn = CrossAttentionBlock(embed_dim, num_heads, dropout)

    def forward(self, image_features, text_features, text_padding_mask=None):
        # Same shared weights for both directions — a compromise, since the
        # optimal attention pattern for image->text differs from text->image
        img_attended, attn_i2t = self.shared_cross_attn(
            query=image_features, key_value=text_features,
            key_padding_mask=text_padding_mask)
        txt_attended, attn_t2i = self.shared_cross_attn(
            query=text_features, key_value=image_features)
        return img_attended, txt_attended, attn_i2t, attn_t2i

### Full VQA Models

Pipeline: encode image + text → cross-modal fusion → mean-pool → concatenate → MLP classifier

The two models are identical except for the fusion layer.

In [16]:
def pool_image_tokens(image_features):
    """Mean-pool spatial image tokens, excluding CLIP/ViT CLS."""
    return image_features[:, 1:, :].mean(dim=1)


def pool_text_tokens(text_features, attention_mask):
    """Masked mean-pool non-padding RoBERTa tokens."""
    mask = attention_mask.unsqueeze(-1).type_as(text_features)
    denom = mask.sum(dim=1).clamp(min=1.0)
    return (text_features * mask).sum(dim=1) / denom


class AsymmetricVQAModel(nn.Module):
    """VQA model with asymmetric (two independent blocks) cross-attention.
    Accepts pre-extracted encoder features — no internal encoders."""

    def __init__(self, num_answers, embed_dim=512, num_heads=8, dropout=0.3):
        super().__init__()
        self.fusion = AsymmetricCrossModalFusion(embed_dim, num_heads, dropout)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(embed_dim, num_answers))

    def forward(self, image_features, text_features, attention_mask):
        text_pad_mask = attention_mask == 0
        img_att, txt_att, attn_i2t, attn_t2i = self.fusion(
            image_features, text_features, text_pad_mask)
        z = torch.cat([
            pool_image_tokens(img_att),
            pool_text_tokens(txt_att, attention_mask),
        ], dim=-1)
        logits = self.classifier(z)
        return logits, {"img_to_txt": attn_i2t, "txt_to_img": attn_t2i}


class SymmetricVQAModel(nn.Module):
    """VQA model with symmetric (single shared block) cross-attention (baseline).
    Accepts pre-extracted encoder features — no internal encoders."""

    def __init__(self, num_answers, embed_dim=512, num_heads=8, dropout=0.3):
        super().__init__()
        self.fusion = SymmetricCrossModalFusion(embed_dim, num_heads, dropout)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(embed_dim, num_answers))

    def forward(self, image_features, text_features, attention_mask):
        text_pad_mask = attention_mask == 0
        img_att, txt_att, attn_i2t, attn_t2i = self.fusion(
            image_features, text_features, text_pad_mask)
        z = torch.cat([
            pool_image_tokens(img_att),
            pool_text_tokens(txt_att, attention_mask),
        ], dim=-1)
        logits = self.classifier(z)
        return logits, {"img_to_txt": attn_i2t, "txt_to_img": attn_t2i}


# --- End-to-end models (used when FREEZE_ENCODERS=False) ---

class AsymmetricVQAModelE2E(nn.Module):
    """End-to-end VQA model: encoders + asymmetric fusion + classifier."""

    def __init__(self, num_answers, embed_dim=512, num_heads=8, dropout=0.3,
                 freeze_encoders=False):
        super().__init__()
        self.image_encoder = ImageEncoder(embed_dim, freeze=freeze_encoders)
        self.text_encoder = TextEncoder(embed_dim, freeze=freeze_encoders)
        self.fusion = AsymmetricCrossModalFusion(embed_dim, num_heads, dropout)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(embed_dim, num_answers))

    def forward(self, images, input_ids, attention_mask):
        img = self.image_encoder(images)
        txt = self.text_encoder(input_ids, attention_mask)
        text_pad_mask = attention_mask == 0
        img_att, txt_att, attn_i2t, attn_t2i = self.fusion(img, txt, text_pad_mask)
        z = torch.cat([
            pool_image_tokens(img_att),
            pool_text_tokens(txt_att, attention_mask),
        ], dim=-1)
        logits = self.classifier(z)
        return logits, {"img_to_txt": attn_i2t, "txt_to_img": attn_t2i}


class SymmetricVQAModelE2E(nn.Module):
    """End-to-end VQA model: encoders + symmetric fusion + classifier."""

    def __init__(self, num_answers, embed_dim=512, num_heads=8, dropout=0.3,
                 freeze_encoders=False):
        super().__init__()
        self.image_encoder = ImageEncoder(embed_dim, freeze=freeze_encoders)
        self.text_encoder = TextEncoder(embed_dim, freeze=freeze_encoders)
        self.fusion = SymmetricCrossModalFusion(embed_dim, num_heads, dropout)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(embed_dim, num_answers))

    def forward(self, images, input_ids, attention_mask):
        img = self.image_encoder(images)
        txt = self.text_encoder(input_ids, attention_mask)
        text_pad_mask = attention_mask == 0
        img_att, txt_att, attn_i2t, attn_t2i = self.fusion(img, txt, text_pad_mask)
        z = torch.cat([
            pool_image_tokens(img_att),
            pool_text_tokens(txt_att, attention_mask),
        ], dim=-1)
        logits = self.classifier(z)
        return logits, {"img_to_txt": attn_i2t, "txt_to_img": attn_t2i}


### Offline Feature Extraction

If `FREEZE_ENCODERS=True`, encoder outputs can still be precomputed. For the upgraded default (`FREEZE_ENCODERS=False`), training uses raw 336px images/tokens and partial end-to-end fine-tuning.


In [17]:
FEATURES_H5 = DATA_DIR / f"vqa_precomputed_features_{RUN_TAG}.h5"

if FREEZE_ENCODERS:
    @torch.no_grad()
    def extract_and_save_features(image_encoder, text_encoder, dataset, split_name, h5_path,
                                   batch_size=BATCH_SIZE, num_workers=NUM_WORKERS):
        """Run frozen encoders once over the dataset and save features to HDF5."""
        image_encoder.eval()
        text_encoder.eval()

        # Use deterministic val transform for reproducible extraction
        orig_transform = dataset.transform
        dataset.transform = get_image_transform("val")

        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                            num_workers=num_workers, pin_memory=True)

        n = len(dataset)
        use_amp = USE_AMP and device.type == "cuda"

        # Estimate disk usage
        bytes_per_sample = (IMAGE_SEQ_LEN * EMBED_DIM + MAX_QUESTION_LEN * EMBED_DIM) * 2  # float16
        bytes_per_sample += MAX_QUESTION_LEN * 1 + NUM_ANSWERS * 4                         # int8 + float32
        est_gb = n * bytes_per_sample / 1e9
        print(f"  Extracting {split_name}: {n:,} samples, estimated {est_gb:.1f} GB")

        with h5py.File(h5_path, "a") as h5f:
            img_ds  = h5f.create_dataset(f"{split_name}/image_features",
                                         shape=(n, IMAGE_SEQ_LEN, EMBED_DIM), dtype="float16")
            txt_ds  = h5f.create_dataset(f"{split_name}/text_features",
                                         shape=(n, MAX_QUESTION_LEN, EMBED_DIM), dtype="float16")
            mask_ds = h5f.create_dataset(f"{split_name}/attention_mask",
                                         shape=(n, MAX_QUESTION_LEN), dtype="int8")
            ans_ds  = h5f.create_dataset(f"{split_name}/answer_target",
                                         shape=(n, NUM_ANSWERS), dtype="float32")

            idx = 0
            for images, input_ids, attention_mask, answers in tqdm(loader, desc=f"  {split_name}"):
                images = images.to(device)
                input_ids = input_ids.to(device)
                attn_mask_dev = attention_mask.to(device)

                amp_ctx = autocast(device_type="cuda", dtype=AMP_DTYPE) if use_amp else nullcontext()
                with amp_ctx:
                    img_feats = image_encoder(images)                       # (B, IMAGE_SEQ_LEN, 512)
                    txt_feats = text_encoder(input_ids, attn_mask_dev)      # (B, MAX_QUESTION_LEN, 512)

                bs = images.size(0)
                img_ds[idx:idx+bs]  = img_feats.cpu().half().numpy()
                txt_ds[idx:idx+bs]  = txt_feats.cpu().half().numpy()
                mask_ds[idx:idx+bs] = attention_mask.numpy().astype("int8")
                ans_ds[idx:idx+bs]  = answers.numpy()
                idx += bs

        dataset.transform = orig_transform
        print(f"  {split_name}: {n:,} samples saved.")

    # --- Run extraction ---
    set_seed(SEED)
    _img_enc = ImageEncoder(EMBED_DIM, freeze=True).to(device)
    _txt_enc = TextEncoder(EMBED_DIM, freeze=True).to(device)

    if FEATURES_H5.exists():
        FEATURES_H5.unlink()

    extract_and_save_features(_img_enc, _txt_enc, train_ds, "train", FEATURES_H5)
    extract_and_save_features(_img_enc, _txt_enc, val_ds,   "val",   FEATURES_H5)

    del _img_enc, _txt_enc
    torch.cuda.empty_cache()

    # Verify
    with h5py.File(FEATURES_H5, "r") as f:
        for split in ["train", "val"]:
            print(f"  {split}: image_features={f[f'{split}/image_features'].shape}, "
                  f"text_features={f[f'{split}/text_features'].shape}")
else:
    print("FREEZE_ENCODERS=False: skipping offline feature extraction.")
    print("Training will use raw 336px images/tokens with end-to-end models.")


FREEZE_ENCODERS=False: skipping offline feature extraction.
Training will use raw 336px images/tokens with end-to-end models.


### Precomputed Feature DatasetReads pre-extracted encoder features directly from HDF5.Bypasses JPEG loading, PIL transforms, and RoBERTa tokenization entirely.

In [18]:
if FREEZE_ENCODERS:
    class PrecomputedVQADataset(Dataset):
        """Dataset that reads pre-extracted encoder features from HDF5."""

        def __init__(self, h5_path, split):
            self.h5_path = str(h5_path)
            self.split = split
            self._h5f = None  # lazy-opened per DataLoader worker

            with h5py.File(self.h5_path, "r") as f:
                self.n = f[f"{split}/image_features"].shape[0]

        def __len__(self):
            return self.n

        def __getitem__(self, idx):
            if self._h5f is None:
                self._h5f = h5py.File(self.h5_path, "r")

            img_feats = torch.from_numpy(
                self._h5f[f"{self.split}/image_features"][idx].astype("float32"))   # (IMAGE_SEQ_LEN, 512)
            txt_feats = torch.from_numpy(
                self._h5f[f"{self.split}/text_features"][idx].astype("float32"))    # (20, 512)
            att_mask = torch.from_numpy(
                self._h5f[f"{self.split}/attention_mask"][idx].astype("int64"))     # (20,)
            answer = torch.from_numpy(
                self._h5f[f"{self.split}/answer_target"][idx])                      # (NUM_ANSWERS,)

            return img_feats, txt_feats, att_mask, answer

    # Shadow the original datasets and loaders with precomputed versions
    train_ds = PrecomputedVQADataset(FEATURES_H5, "train")
    val_ds   = PrecomputedVQADataset(FEATURES_H5, "val")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

    print(f"Train: {len(train_ds):,} samples ({len(train_loader)} batches)")
    print(f"Val:   {len(val_ds):,} samples ({len(val_loader)} batches)")

    # Sanity check: verify shapes from one batch
    _img, _txt, _mask, _ans = next(iter(val_loader))
    print(f"Batch shapes: img={_img.shape}, txt={_txt.shape}, mask={_mask.shape}, ans={_ans.shape}")
    del _img, _txt, _mask, _ans
else:
    # Rebuild loaders with the (smaller) BATCH_SIZE for end-to-end training
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

    print(f"End-to-end mode (batch_size={BATCH_SIZE})")
    print(f"Train: {len(train_ds):,} samples ({len(train_loader)} batches)")
    print(f"Val:   {len(val_ds):,} samples ({len(val_loader)} batches)")

End-to-end mode (batch_size=32)
Train: 435,080 samples (13597 batches)
Val:   209,782 samples (6556 batches)


---
## 3. Training

In [19]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, use_amp, scheduler=None):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    optimizer.zero_grad(set_to_none=True)
    for step, (inp1, inp2, masks, answers) in enumerate(tqdm(loader, desc="  train", leave=False), start=1):
        inp1    = inp1.to(device)
        inp2    = inp2.to(device)
        masks   = masks.to(device)
        answers = answers.to(device)

        amp_ctx = autocast(device_type="cuda", dtype=AMP_DTYPE) if use_amp else nullcontext()
        with amp_ctx:
            logits, _ = model(inp1, inp2, masks)
            loss = criterion(logits, answers)

        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite loss at training step {step}: {loss.item()}")

        loss_for_backward = loss / GRAD_ACCUM_STEPS
        if scaler is not None:
            scaler.scale(loss_for_backward).backward()
        else:
            loss_for_backward.backward()

        should_step = (step % GRAD_ACCUM_STEPS == 0) or (step == len(loader))
        if should_step:
            if scaler is not None:
                scaler.unscale_(optimizer)
            if MAX_GRAD_NORM is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            if scheduler is not None:
                scheduler.step()

        total_loss += loss.item() * answers.size(0)
        correct += (logits.argmax(dim=1) == answers.argmax(dim=1)).sum().item()
        total += answers.size(0)

    return {"train_loss": total_loss / total, "train_acc": correct / total * 100}


@torch.no_grad()
def evaluate(model, loader, criterion, use_amp):
    """Evaluate with target weights already encoded as VQA scores in [0, 1]."""
    model.eval()
    total_loss = 0.0
    vqa_acc_sum = 0.0
    vqa_acc_top1000_sum = 0.0
    total = 0

    for inp1, inp2, masks, answers in tqdm(loader, desc="  eval", leave=False):
        inp1    = inp1.to(device)
        inp2    = inp2.to(device)
        masks   = masks.to(device)
        answers = answers.to(device)

        amp_ctx = autocast(device_type="cuda", dtype=AMP_DTYPE) if use_amp else nullcontext()
        with amp_ctx:
            logits, _ = model(inp1, inp2, masks)
            loss = criterion(logits, answers)

        total_loss += loss.item() * answers.size(0)

        preds = logits.argmax(dim=1)
        vqa_scores = answers[torch.arange(answers.size(0), device=answers.device), preds]
        vqa_acc_sum += vqa_scores.sum().item()

        if REPORT_TOP1000_COMPAT:
            n_top = min(1000, logits.size(1))
            preds_top1000 = logits[:, :n_top].argmax(dim=1)
            top1000_scores = answers[torch.arange(answers.size(0), device=answers.device), preds_top1000]
            vqa_acc_top1000_sum += top1000_scores.sum().item()

        total += answers.size(0)

    metrics = {
        "val_loss": total_loss / total,
        "val_vqa_acc": vqa_acc_sum / total * 100,
    }
    if REPORT_TOP1000_COMPAT:
        metrics["val_vqa_acc_top1000"] = vqa_acc_top1000_sum / total * 100
    return metrics


In [20]:
def make_model(model_type):
    if FREEZE_ENCODERS:
        if model_type == "asymmetric":
            return AsymmetricVQAModel(NUM_ANSWERS, EMBED_DIM, NUM_HEADS, DROPOUT)
        return SymmetricVQAModel(NUM_ANSWERS, EMBED_DIM, NUM_HEADS, DROPOUT)

    if model_type == "asymmetric":
        return AsymmetricVQAModelE2E(NUM_ANSWERS, EMBED_DIM, NUM_HEADS, DROPOUT,
                                    freeze_encoders=False)
    return SymmetricVQAModelE2E(NUM_ANSWERS, EMBED_DIM, NUM_HEADS, DROPOUT,
                                freeze_encoders=False)


def build_optimizer_and_scheduler(model, total_training_steps=None):
    """Create optimizer groups: low LR encoders, higher LR projections/fusion/classifier."""
    if FREEZE_ENCODERS:
        params = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        return optimizer, None, [("trainable", sum(p.numel() for p in params), LEARNING_RATE)]

    encoder_params = []
    head_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith("image_encoder.clip") or name.startswith("text_encoder.roberta"):
            encoder_params.append(param)
        else:
            head_params.append(param)

    optimizer = torch.optim.AdamW([
        {"params": encoder_params, "lr": ENCODER_LR},
        {"params": head_params,    "lr": LEARNING_RATE},
    ], weight_decay=WEIGHT_DECAY)

    steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
    if total_training_steps is None:
        total_training_steps = max(1, steps_per_epoch * EPOCHS)
    warmup_steps = max(0, steps_per_epoch * WARMUP_EPOCHS)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=max(1, total_training_steps),
    )
    group_counts = [
        ("encoders", sum(p.numel() for p in encoder_params), ENCODER_LR),
        ("projection+fusion+classifier", sum(p.numel() for p in head_params), LEARNING_RATE),
    ]
    return optimizer, scheduler, group_counts


def run_training(model_type, run_name=None, resume_from=None):
    """Train a VQA model or resume from a same-run-tag checkpoint."""
    set_seed(SEED)
    if run_name is None:
        run_name = f"{RUN_TAG}_{model_type}"

    model = make_model(model_type).to(device)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\nModel: {model_type} | Run: {run_name}")
    print(f"  Trainable params: {trainable:,} / {total_params:,} total")

    criterion = nn.BCEWithLogitsLoss()
    optimizer, scheduler, group_counts = build_optimizer_and_scheduler(model)
    for group_name, count, lr in group_counts:
        print(f"  {group_name}: {count:,} params (lr={lr})")

    use_amp = USE_AMP and device.type == "cuda"
    scaler = GradScaler("cuda", enabled=(use_amp and AMP_DTYPE == torch.float16)) if use_amp else None

    history = []
    best_val_acc = 0.0
    best_acc_epoch = None
    start_epoch = 1

    # --- RESUME LOGIC: only same RUN_TAG checkpoints are selected by caller cells. ---
    if resume_from is not None:
        checkpoint_path = Path(resume_from)
        if checkpoint_path.exists():
            print(f"Resuming from checkpoint: {checkpoint_path}")
            checkpoint = torch.load(checkpoint_path, map_location=device)

            model.load_state_dict(checkpoint["model_state_dict"])
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            if scheduler is not None and checkpoint.get("scheduler_state_dict") is not None:
                scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
            start_epoch = checkpoint["epoch"] + 1

            history_file = METRICS_DIR / f"{run_name}_history.json"
            if history_file.exists():
                with open(history_file, "r") as f:
                    history = json.load(f)

            if history:
                best_entry = max(history, key=lambda h: h.get("val_vqa_acc", 0))
                best_val_acc = best_entry.get("val_vqa_acc", 0)
                best_acc_epoch = best_entry["epoch"]
        else:
            print(f"Checkpoint not found at {checkpoint_path}. Starting from scratch.")

    # --- TRAINING LOOP ---
    for epoch in range(start_epoch, EPOCHS + 1):
        t0 = time.time()
        train_m = train_one_epoch(model, train_loader, criterion, optimizer, scaler, use_amp,
                                  scheduler=scheduler)
        val_m = evaluate(model, val_loader, criterion, use_amp)
        elapsed = time.time() - t0

        epoch_data = {"epoch": epoch, **train_m, **val_m, "elapsed_s": round(elapsed, 1)}
        history.append(epoch_data)

        compat = ""
        if "val_vqa_acc_top1000" in val_m:
            compat = f" | top1000 {val_m['val_vqa_acc_top1000']:.2f}%"
        print(f"  Epoch {epoch}/{EPOCHS} | loss {train_m['train_loss']:.4f} | "
              f"train {train_m['train_acc']:.2f}% | vqa_acc {val_m['val_vqa_acc']:.2f}%"
              f"{compat} | {elapsed:.0f}s")

        epoch_ckpt = CHECKPOINT_DIR / f"{run_name}_epoch{epoch}.pt"
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "metrics": epoch_data,
            "config": {
                "run_tag": RUN_TAG,
                "image_encoder": IMAGE_ENCODER_NAME,
                "image_size": IMAGE_SIZE,
                "num_answers": NUM_ANSWERS,
                "target": "vqa_score_min_count_over_3",
            },
        }, epoch_ckpt)

        if val_m["val_vqa_acc"] > best_val_acc:
            best_val_acc = val_m["val_vqa_acc"]
            best_acc_epoch = epoch
            torch.save({"model_state_dict": model.state_dict(), **epoch_data},
                       CHECKPOINT_DIR / f"{run_name}_best.pt")
            print(f"    New best: {best_val_acc:.2f}%")

        keep = {epoch_ckpt}
        if best_acc_epoch is not None:
            keep.add(CHECKPOINT_DIR / f"{run_name}_epoch{best_acc_epoch}.pt")
        for stale in CHECKPOINT_DIR.glob(f"{run_name}_epoch*.pt"):
            if stale not in keep:
                stale.unlink(missing_ok=True)

        with open(METRICS_DIR / f"{run_name}_history.json", "w") as f:
            json.dump(history, f, indent=2)

    print(f"Training complete. Best val_vqa_acc: {best_val_acc:.2f}%")
    return model, history


In [21]:
# --- Smoke tests before full training ---
def smoke_test_model(model_type="asymmetric", max_steps=SMOKE_TEST_STEPS):
    """Check feature shapes, attention grid size, and NaN-free short training."""
    if max_steps <= 0:
        print("Smoke test skipped because SMOKE_TEST_STEPS <= 0.")
        return

    print(f"Running {model_type} smoke test for up to {max_steps} mini-batches...")
    set_seed(SEED)
    model = make_model(model_type).to(device)
    model.train()

    criterion = nn.BCEWithLogitsLoss()
    total_optimizer_steps = max(1, math.ceil(max_steps / GRAD_ACCUM_STEPS))
    optimizer, scheduler, _ = build_optimizer_and_scheduler(
        model, total_training_steps=total_optimizer_steps)
    use_amp = USE_AMP and device.type == "cuda"
    scaler = GradScaler("cuda", enabled=(use_amp and AMP_DTYPE == torch.float16)) if use_amp else None

    optimizer.zero_grad(set_to_none=True)
    steps_run = 0
    loader_iter = iter(train_loader)

    for step in tqdm(range(1, max_steps + 1), desc="smoke", leave=False):
        try:
            inp1, inp2, masks, answers = next(loader_iter)
        except StopIteration:
            break

        inp1    = inp1.to(device)
        inp2    = inp2.to(device)
        masks   = masks.to(device)
        answers = answers.to(device)

        if step == 1:
            with torch.no_grad():
                img_feats = model.image_encoder(inp1)
            assert img_feats.shape[1] == IMAGE_SEQ_LEN, img_feats.shape
            assert img_feats.shape[2] == EMBED_DIM, img_feats.shape
            print(f"  image features: {tuple(img_feats.shape)}")

        amp_ctx = autocast(device_type="cuda", dtype=AMP_DTYPE) if use_amp else nullcontext()
        with amp_ctx:
            logits, attn = model(inp1, inp2, masks)
            loss = criterion(logits, answers)

        if step == 1:
            grid_size = int(math.sqrt(attn["txt_to_img"].shape[-1] - 1))
            assert grid_size == IMAGE_SIZE // IMAGE_PATCH_SIZE, grid_size
            print(f"  attention grid: {grid_size}x{grid_size}")

        if not torch.isfinite(loss):
            raise FloatingPointError(f"Smoke test found non-finite loss at step {step}: {loss.item()}")

        loss_for_backward = loss / GRAD_ACCUM_STEPS
        if scaler is not None:
            scaler.scale(loss_for_backward).backward()
        else:
            loss_for_backward.backward()

        should_step = (step % GRAD_ACCUM_STEPS == 0) or (step == max_steps)
        if should_step:
            if scaler is not None:
                scaler.unscale_(optimizer)
            if MAX_GRAD_NORM is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            if scheduler is not None:
                scheduler.step()

        steps_run += 1

    del model, optimizer, scheduler
    if device.type == "cuda":
        torch.cuda.empty_cache()
    print(f"Smoke test passed: {steps_run} mini-batches, no NaNs.")


if RUN_SMOKE_TEST:
    smoke_test_model("asymmetric")
else:
    print("RUN_SMOKE_TEST=False: skipping smoke test.")


Running asymmetric smoke test for up to 200 mini-batches...


config.json:   0%|          | 0.00/4.76k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-large-patch14-336
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.se

...

  image features: (32, 577, 512)


/tmp/ipykernel_499/3797670793.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  amp_ctx = autocast() if use_amp else nullcontext()


  attention grid: 24x24



smoke: 100%|██████████| 200/200 [00:48<00:00,  4.85it/s]
                                                        

Smoke test passed: 200 mini-batches, no NaNs.


### 3.1 Train Symmetric Baseline

In [22]:
# Auto-resume only same-run-tag symmetric checkpoints, never older ViT-B/top-1000 checkpoints.
sym_ckpts = sorted(
    CHECKPOINT_DIR.glob(f"{RUN_TAG}_symmetric_epoch*.pt"),
    key=lambda p: int(p.stem.rsplit("epoch", 1)[1]),
)
sym_resume = sym_ckpts[-1] if sym_ckpts else None
symmetric_model, symmetric_history = run_training("symmetric", resume_from=sym_resume)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-large-patch14-336
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.se

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Model: symmetric | Run: clipL336_vqaScore_top3000_s42_symmetric
  Trainable params: 181,168,312 / 434,288,824 total
  encoders: 175,032,576 params (lr=2e-06)
  projection+fusion+classifier: 6,135,736 params (lr=0.0002)
Resuming from checkpoint: /content/results/checkpoints/clipL336_vqaScore_top3000_s42_symmetric_epoch7.pt


...

  Epoch 9/13 | loss 0.0009 | train 68.84% | vqa_acc 71.72% | top1000 70.79% | 3922s
    New best: 71.72%


  Epoch 10/13 | loss 0.0008 | train 69.92% | vqa_acc 72.00% | top1000 71.05% | 3914s
    New best: 72.00%


  Epoch 11/13 | loss 0.0008 | train 71.03% | vqa_acc 72.20% | top1000 71.24% | 3926s
    New best: 72.20%


  Epoch 12/13 | loss 0.0008 | train 71.89% | vqa_acc 72.25% | top1000 71.27% | 3917s
    New best: 72.25%


  Epoch 13/13 | loss 0.0008 | train 72.14% | vqa_acc 72.26% | top1000 71.28% | 3926s
    New best: 72.26%
Training complete. Best val_vqa_acc: 72.26%


In [23]:
!mkdir -p /content/drive/MyDrive/unfrozen_results_ViT_CLIP_RoBERTa/

!cp -r /content/results/ /content/drive/MyDrive/unfrozen_results_ViT_CLIP_RoBERTa/

### 3.2 Train Asymmetric Model

In [ ]:
# Auto-resume only same-run-tag asymmetric checkpoints, never older ViT-B/top-1000 checkpoints.
asym_ckpts = sorted(
    CHECKPOINT_DIR.glob(f"{RUN_TAG}_asymmetric_epoch*.pt"),
    key=lambda p: int(p.stem.rsplit("epoch", 1)[1]),
)
asym_resume = asym_ckpts[-1] if asym_ckpts else None
asymmetric_model, asymmetric_history = run_training("asymmetric", resume_from=asym_resume)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-large-patch14-336
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.se

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_499/1756705660.py:71: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if use_amp else None



Model: asymmetric | Run: clipL336_vqaScore_top3000_s42_asymmetric
  Trainable params: 184,321,720 / 437,442,232 total
  encoders: 175,032,576 params (lr=2e-06)
  projection+fusion+classifier: 9,289,144 params (lr=0.0002)


...

  Epoch 2/13 | loss 0.0014 | train 44.68% | vqa_acc 56.82% | top1000 56.55% | 3778s
    New best: 56.82%


  Epoch 3/13 | loss 0.0012 | train 49.59% | vqa_acc 60.14% | top1000 59.73% | 3786s
    New best: 60.14%


  Epoch 4/13 | loss 0.0012 | train 53.18% | vqa_acc 62.21% | top1000 61.72% | 3803s
    New best: 62.21%


  Epoch 5/13 | loss 0.0011 | train 56.34% | vqa_acc 64.06% | top1000 63.49% | 3788s
    New best: 64.06%


  train:   0%|          | 34/13597 [00:09<48:53,  4.62it/s]

In [ ]:
!mkdir -p /content/drive/MyDrive/unfrozen_results_ViT_CLIP_RoBERTa/

!cp -r /content/results/ /content/drive/MyDrive/unfrozen_results_ViT_CLIP_RoBERTa/

### 3.3 Load Models from .pt File

In [ ]:
# Rebuild the model architecture
if FREEZE_ENCODERS:
    symmetric_model = SymmetricVQAModel(NUM_ANSWERS, EMBED_DIM, NUM_HEADS, DROPOUT).to(device)
else:
    symmetric_model = SymmetricVQAModelE2E(NUM_ANSWERS, EMBED_DIM, NUM_HEADS, DROPOUT,
                                            freeze_encoders=False).to(device)

sym_checkpoint = torch.load(CHECKPOINT_DIR / f"{RUN_TAG}_symmetric_best.pt", map_location=device)
symmetric_model.load_state_dict(sym_checkpoint["model_state_dict"])
symmetric_model.eval()

# Load its history so the graphs plot correctly
sym_history_file = METRICS_DIR / f"{RUN_TAG}_symmetric_history.json"
if sym_history_file.exists():
    with open(sym_history_file, "r") as f:
        symmetric_history = json.load(f)
else:
    symmetric_history = []
    print(f"WARNING: {sym_history_file.name} missing — training did not complete.")


In [ ]:
# Rebuild the model architecture
if FREEZE_ENCODERS:
    asymmetric_model = AsymmetricVQAModel(NUM_ANSWERS, EMBED_DIM, NUM_HEADS, DROPOUT).to(device)
else:
    asymmetric_model = AsymmetricVQAModelE2E(NUM_ANSWERS, EMBED_DIM, NUM_HEADS, DROPOUT,
                                              freeze_encoders=False).to(device)

asym_checkpoint = torch.load(CHECKPOINT_DIR / f"{RUN_TAG}_asymmetric_best.pt", map_location=device)
asymmetric_model.load_state_dict(asym_checkpoint["model_state_dict"])
asymmetric_model.eval()

# Load its history so the graphs plot correctly
asym_history_file = METRICS_DIR / f"{RUN_TAG}_asymmetric_history.json"
if asym_history_file.exists():
    with open(asym_history_file, "r") as f:
        asymmetric_history = json.load(f)
else:
    asymmetric_history = []
    print(f"WARNING: {asym_history_file.name} missing — training did not complete.")


---
## 4. Evaluation

In [ ]:
criterion = nn.BCEWithLogitsLoss()
use_amp = USE_AMP and device.type == "cuda"

sym_metrics = evaluate(symmetric_model, val_loader, criterion, use_amp)
asym_metrics = evaluate(asymmetric_model, val_loader, criterion, use_amp)

results = {"Symmetric": sym_metrics, "Asymmetric": asym_metrics}

# Print comparison table
metrics_keys = ["val_vqa_acc", "val_vqa_acc_top1000", "val_loss"]
header = "| Method       | " + " | ".join(k.replace('_', ' ').title() for k in metrics_keys) + " |"
sep    = "|--------------|" + "|".join("----------" for _ in metrics_keys) + "|"
print(header)
print(sep)
for name, vals in results.items():
    row = f"| {name:<12} | " + " | ".join(f"{vals.get(k, float('nan')):>8.2f}" for k in metrics_keys) + " |"
    print(row)


---
## 5. Visualization

### 5.1 Training Curves

In [ ]:
histories = {"Symmetric": symmetric_history, "Asymmetric": asymmetric_history}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, hist in histories.items():
    epochs = [h["epoch"] for h in hist]
    axes[0].plot(epochs, [h["train_loss"] for h in hist], label=f"{name} train")
    axes[0].plot(epochs, [h["val_loss"] for h in hist], "--", label=f"{name} val")
    axes[1].plot(epochs, [h["train_acc"] for h in hist], label=f"{name} train")
    axes[1].plot(epochs, [h["val_vqa_acc"] for h in hist], "--", label=f"{name} val")

axes[0].set(xlabel="Epoch", ylabel="Loss", title="Loss")
axes[1].set(xlabel="Epoch", ylabel="Accuracy (%)", title="VQA Accuracy")
for ax in axes:
    ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.2 Comparison Bar Chart

In [ ]:
metric_names = ["val_vqa_acc"]
model_names = list(results.keys())
x = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(6, 5))
for i, name in enumerate(model_names):
    values = [results[name][m] for m in metric_names]
    bars = ax.bar(x + i * width, values, width, label=name)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{val:.1f}", ha="center", va="bottom", fontsize=10)

ax.set_xticks(x + width / 2)
ax.set_xticklabels([m.replace("_", " ").title() for m in metric_names])
ax.set_ylabel("Accuracy (%)")
ax.set_title("Model Comparison — VQA Accuracy")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "comparison_bar.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
!mkdir -p /content/drive/MyDrive/unfrozen_results_ViT_CLIP_RoBERTa/

!cp -r /content/results/ /content/drive/MyDrive/unfrozen_results_ViT_CLIP_RoBERTa/

### Visualization PreparationRe-attach frozen encoders to trained fusion models and reload the raw datasetfor visualization. Training used precomputed features; visualization needs rawimages and question strings.

In [ ]:
# Re-instantiate the raw dataset for visualization (images + question strings)
raw_val_ds = VQADataset(
    questions_file=DATA_DIR / "questions" / "v2_OpenEnded_mscoco_val2014_questions.json",
    annotations_file=DATA_DIR / "answers" / "v2_mscoco_val2014_annotations.json",
    h5_path=h5_path,
    answer_to_idx=answer_to_idx,
    max_question_len=MAX_QUESTION_LEN,
    transform=get_image_transform("val"),
    max_samples=MAX_SAMPLES,
)

raw_val_loader = DataLoader(raw_val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)

if FREEZE_ENCODERS:
    class EndToEndVQAWrapper(nn.Module):
        """Wraps a trained fusion model with fresh frozen encoders for visualization."""

        def __init__(self, trained_fusion_model, embed_dim=EMBED_DIM):
            super().__init__()
            self.image_encoder = ImageEncoder(embed_dim, freeze=True)
            self.text_encoder = TextEncoder(embed_dim, freeze=True)
            self.trained_model = trained_fusion_model

        def forward(self, images, input_ids, attention_mask):
            img_feats = self.image_encoder(images)
            txt_feats = self.text_encoder(input_ids, attention_mask)
            return self.trained_model(img_feats, txt_feats, attention_mask)

    set_seed(SEED)
    e2e_asymmetric = EndToEndVQAWrapper(asymmetric_model).to(device)
    e2e_symmetric = EndToEndVQAWrapper(symmetric_model).to(device)
    e2e_models_dict = {"Asymmetric": e2e_asymmetric, "Symmetric": e2e_symmetric}
else:
    # Models are already end-to-end — use them directly
    e2e_models_dict = {"Asymmetric": asymmetric_model, "Symmetric": symmetric_model}

print(f"raw_val_ds: {len(raw_val_ds):,} samples")
print("End-to-end models ready for visualization.")

### 5.3 Attention Heatmaps

Visualize what the asymmetric model attends to: which image regions light up for each question word, and which words are most important for each image patch.

In [ ]:
# Visualization helpers
MEAN = np.array(CLIP_MEAN)
STD  = np.array(CLIP_STD)


def denormalize(img_tensor):
    """Convert normalised (3,H,W) tensor to (H,W,3) uint8 array."""
    img = img_tensor.cpu().numpy().transpose(1, 2, 0)
    img = img * STD + MEAN
    return np.clip(img * 255, 0, 255).astype(np.uint8)


def decode_tokens(input_ids):
    """Decode token IDs to readable strings."""
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
    return [tokenizer.decode(tid) for tid in input_ids.tolist()]


@torch.no_grad()
def get_attention_weights(model, image, input_ids, attention_mask):
    """Run a single sample and return attention weight dicts."""
    model.eval()
    img = image.unsqueeze(0).to(device)
    ids = input_ids.unsqueeze(0).to(device)
    mask = attention_mask.unsqueeze(0).to(device)
    _, attn = model(img, ids, mask)
    return {k: v.cpu() for k, v in attn.items()}


def plot_image_attention(attn_t2i, image_tensor, tokens, question, top_tokens=4):
    """Overlay text->image attention heatmaps on the original image."""
    img_np = denormalize(image_tensor)
    attn = attn_t2i.squeeze(0).numpy()  # (N_txt, N_img)

    grid_size = int(np.sqrt(attn.shape[1] - 1))  # 24 for CLIP ViT-L/14@336
    attn_spatial = attn[:, 1:]  # drop CLS column
    image_wh = (img_np.shape[1], img_np.shape[0])

    # Count how many actual words exist (ignoring padding)
    real_token_count = len([t for t in tokens if t not in ['<pad>']])

    # Only average the attention maps of the real tokens
    combined = attn_spatial[:real_token_count].mean(axis=0).reshape(grid_size, grid_size)

    combined_resized = np.array(
        Image.fromarray(combined).resize(image_wh, Image.BILINEAR))

    token_importance = attn_spatial.sum(axis=1)
    top_idx = token_importance.argsort()[-top_tokens:][::-1]

    n_cols = min(top_tokens, len(top_idx)) + 1
    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))
    if n_cols == 1:
        axes = [axes]

    axes[0].imshow(img_np)
    axes[0].imshow(combined_resized, alpha=0.5, cmap="jet")
    axes[0].set_title("Combined")
    axes[0].axis("off")

    for i, idx in enumerate(top_idx):
        if i + 1 >= len(axes):
            break
        token_attn = attn_spatial[idx].reshape(grid_size, grid_size)
        token_resized = np.array(
            Image.fromarray(token_attn).resize(image_wh, Image.BILINEAR))
        label = tokens[idx] if tokens else f"token {idx}"
        axes[i + 1].imshow(img_np)
        axes[i + 1].imshow(token_resized, alpha=0.5, cmap="jet")
        axes[i + 1].set_title(f'"{label}"')
        axes[i + 1].axis("off")

    fig.suptitle(question, fontsize=12)
    fig.tight_layout()
    return fig


def plot_text_attention(attn_img_to_txt, tokens, question):
    """Bar chart of image->text attention per token."""
    attn = attn_img_to_txt.squeeze(0).numpy()
    token_weights = attn.mean(axis=0)

    fig, ax = plt.subplots(figsize=(6, max(3, len(tokens) * 0.35)))
    y_pos = np.arange(len(tokens))
    ax.barh(y_pos, token_weights, color="steelblue")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(tokens, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel("Mean attention weight")
    ax.set_title(f"Image -> Text attention\n{question}")
    fig.tight_layout()
    return fig


In [ ]:
# Generate attention maps for sample validation images
for i in range(min(5, len(raw_val_ds))):
    image, input_ids, attention_mask, answer_target = raw_val_ds[i]
    question = raw_val_ds.samples[i]["question"]
    tokens = decode_tokens(input_ids)

    attn = get_attention_weights(e2e_models_dict["Asymmetric"], image, input_ids, attention_mask)

    fig = plot_image_attention(attn["txt_to_img"], image, tokens, question)
    fig.savefig(FIGURES_DIR / f"attn_img_{i}.png", dpi=150, bbox_inches="tight")
    plt.show()

    fig = plot_text_attention(attn["img_to_txt"], tokens, question)
    fig.savefig(FIGURES_DIR / f"attn_txt_{i}.png", dpi=150, bbox_inches="tight")
    plt.show()

### 5.4 Qualitative Comparison Grid

Side-by-side predictions and attention maps for both models on the same samples.

In [ ]:
@torch.no_grad()
def qualitative_grid(models, dataset, idx_to_answer, n_samples=6, save_path=None):
    """Grid of qualitative examples comparing models."""
    sample_indices = torch.randperm(len(dataset))[:n_samples].tolist()
    model_names = list(models.keys())
    n_cols = 1 + len(model_names)
    n_rows = len(sample_indices)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    for row, idx in enumerate(sample_indices):
        image, input_ids, attention_mask, answer_target = dataset[idx]
        question = dataset.samples[idx]["question"]
        gt_answer = idx_to_answer[answer_target.argmax().item()]
        img_np = denormalize(image)
        image_wh = (img_np.shape[1], img_np.shape[0])

        # Column 0: original image + question
        axes[row, 0].imshow(img_np)
        axes[row, 0].set_title(f"Q: {question}\nGT: {gt_answer}", fontsize=9)
        axes[row, 0].axis("off")

        # Remaining columns: one per model
        for col, name in enumerate(model_names, start=1):
            model = models[name]
            attn = get_attention_weights(model, image, input_ids, attention_mask)

            img_t = image.unsqueeze(0).to(device)
            ids_t = input_ids.unsqueeze(0).to(device)
            mask_t = attention_mask.unsqueeze(0).to(device)
            logits, _ = model(img_t, ids_t, mask_t)
            pred_answer = idx_to_answer.get(logits.argmax(dim=1).item(), "???")

            # Attention heatmap
            attn_t2i = attn["txt_to_img"].squeeze(0).numpy()
            grid_size = int(np.sqrt(attn_t2i.shape[1] - 1))
            combined = attn_t2i[:, 1:].mean(axis=0).reshape(grid_size, grid_size)
            combined_resized = np.array(
                Image.fromarray(combined).resize(image_wh, Image.BILINEAR))

            axes[row, col].imshow(img_np)
            axes[row, col].imshow(combined_resized, alpha=0.5, cmap="jet")
            marker = "correct" if pred_answer == gt_answer else "wrong"
            axes[row, col].set_title(f"{name}\nPred: {pred_answer} ({marker})", fontsize=9)
            axes[row, col].axis("off")

    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    return fig


In [ ]:
fig = qualitative_grid(
    e2e_models_dict, raw_val_ds, idx_to_answer, n_samples=6,
    save_path=str(FIGURES_DIR / "qualitative_grid.png"))
plt.show()

In [ ]:
@torch.no_grad()
def error_analysis(models, loader, idx_to_answer, skip_n=1):
    print(f"Running Error Analysis... skipping {skip_n} cases")
    asym_model = models["Asymmetric"]
    sym_model = models["Symmetric"]

    asym_model.eval()
    sym_model.eval()

    cases_found = 0

    for images, input_ids, attention_mask, answers in loader:
        images = images.to(device)
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        targets = answers.to(device).argmax(dim=1)

        asym_logits, _ = asym_model(images, input_ids, attention_mask)
        sym_logits, _ = sym_model(images, input_ids, attention_mask)

        asym_preds = asym_logits.argmax(dim=1)
        sym_preds = sym_logits.argmax(dim=1)

        # Find condition: Asymmetric correct AND Symmetric wrong
        mask = (asym_preds == targets) & (sym_preds != targets)
        divergent_indices = mask.nonzero(as_tuple=True)[0]

        for idx in divergent_indices:
            if cases_found < skip_n:
                cases_found += 1
                continue

            print("--- Found a divergent case ---")
            print(f"Target Answer: {idx_to_answer[targets[idx].item()]}")
            print(f"Asymmetric Guess: {idx_to_answer[asym_preds[idx].item()]} (Correct)")
            print(f"Symmetric Guess: {idx_to_answer[sym_preds[idx].item()]} (Wrong)")

            # Extract the specific sample for visualization
            img = images[idx].cpu()
            ids = input_ids[idx].cpu()
            mask_ = attention_mask[idx].cpu()

            # Decode tokens to reconstruct a readable question
            tokens = decode_tokens(ids)
            clean_tokens = [t.replace('\u0120', '') for t in tokens if t not in ['<pad>', '<s>', '</s>']]
            question_str = " ".join(clean_tokens).strip() + "?"
            print(f"Question: {question_str}")

            # Get attention weights for BOTH models
            attn_asym = get_attention_weights(asym_model, img, ids, mask_)
            attn_sym = get_attention_weights(sym_model, img, ids, mask_)

            print("\n=== ASYMMETRIC MODEL ATTENTION (Correct Guess) ===")
            fig_img_asym = plot_image_attention(attn_asym["txt_to_img"], img, tokens, question_str)
            plt.show()
            fig_txt_asym = plot_text_attention(attn_asym["img_to_txt"], tokens, question_str)
            plt.show()

            print("\n=== SYMMETRIC MODEL ATTENTION (Wrong Guess) ===")
            fig_img_sym = plot_image_attention(attn_sym["txt_to_img"], img, tokens, question_str)
            plt.show()
            fig_txt_sym = plot_text_attention(attn_sym["img_to_txt"], tokens, question_str)
            plt.show()

            return

error_analysis(e2e_models_dict, raw_val_loader, idx_to_answer, skip_n=10)

# Task
Categorize the validation questions by type (e.g., 'Is/Are', 'How many', 'What color') and compute the accuracy for both the symmetric and asymmetric models per category, generating a grouped bar chart to visualize the results. Additionally, conduct a comprehensive modality ablation test by evaluating both models under three conditions: Full Data, Image-Blind (zeroed images), and Text-Blind (zeroed input IDs/masks), and generate a grouped bar chart showing the degradation. Finally, provide a brief summary of the insights gained from these new evaluation metrics to help frame the presentation.

## Question Type Accuracy Breakdown

### Subtask:
Categorize validation questions by type, calculate accuracy per category for both models, and plot a grouped bar chart.


In [ ]:
from collections import defaultdict

# 1. Categorize question function
def categorize_question(question):
    q_lower = question.lower().strip()
    if q_lower.startswith(('is ', 'are ', 'was ', 'were ', 'does ', 'do ', 'has ', 'have ', 'can ', 'could ', 'would ', 'should ')):
        return 'Yes/No'
    elif q_lower.startswith('how many'):
        return 'Count'
    elif q_lower.startswith('what color'):
        return 'Color'
    else:
        return 'Other'

@torch.no_grad()
def evaluate_by_question_type(models, loader, dataset):
    """Evaluate VQA accuracy broken down by question type.
    Uses loader for model inference, raw dataset for question text."""
    for model in models.values():
        model.eval()

    category_scores = {name: defaultdict(list) for name in models.keys()}
    global_idx = 0

    use_amp = USE_AMP and device.type == "cuda"

    for inp1, inp2, masks, answers in tqdm(loader, desc="Evaluating by question type"):
        inp1    = inp1.to(device)
        inp2    = inp2.to(device)
        masks   = masks.to(device)
        answers = answers.to(device)

        batch_size = inp1.size(0)

        preds_dict = {}
        for name, model in models.items():
            amp_ctx = autocast(device_type="cuda", dtype=AMP_DTYPE) if use_amp else nullcontext()
            with amp_ctx:
                logits, _ = model(inp1, inp2, masks)
            preds_dict[name] = logits.argmax(dim=1)

        # 3. Calculate category score per sample. Targets already store VQA scores.
        for i in range(batch_size):
            question = dataset.samples[global_idx + i]["question"]
            category = categorize_question(question)

            for name, preds in preds_dict.items():
                pred_idx = preds[i]
                vqa_score = answers[i, pred_idx].item()
                category_scores[name][category].append(vqa_score)

        global_idx += batch_size

    # 4. Aggregate scores
    category_acc = {name: {} for name in models.keys()}
    for name, cat_scores in category_scores.items():
        for cat, scores in cat_scores.items():
            category_acc[name][cat] = np.mean(scores) * 100

    return category_acc

# Use precomputed models + val_loader for speed, raw_val_ds for question text
category_acc = evaluate_by_question_type(
    {"Symmetric": symmetric_model, "Asymmetric": asymmetric_model},
    val_loader, raw_val_ds)

# 5. Generate a grouped bar chart
categories = ['Yes/No', 'Count', 'Color', 'Other']
sym_acc = [category_acc["Symmetric"].get(cat, 0) for cat in categories]
asym_acc = [category_acc["Asymmetric"].get(cat, 0) for cat in categories]

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 6))
bars1 = ax.bar(x - width/2, sym_acc, width, label='Symmetric', color='lightblue')
bars2 = ax.bar(x + width/2, asym_acc, width, label='Asymmetric', color='steelblue')

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=9)

ax.set_ylabel('Accuracy (%)')
ax.set_title('VQA Accuracy by Question Type')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES_DIR / "question_type_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()


## Modality Ablation Test

Evaluate both models under three conditions: Full Data, Image-Blind (zeroed images), and Text-Blind (zeroed input IDs/masks), and generate a grouped bar chart showing the degradation.

In [ ]:
@torch.no_grad()
def evaluate_ablation(model, loader, condition):
    """Evaluate model under ablation conditions."""
    model.eval()
    vqa_acc_sum = 0.0
    total = 0
    use_amp = USE_AMP and device.type == "cuda"

    for inp1, inp2, masks, answers in tqdm(loader, desc=f"Eval {condition}"):
        if condition == "Image-Blind":
            inp1 = torch.zeros_like(inp1)
        elif condition == "Text-Blind":
            if FREEZE_ENCODERS:
                inp2 = torch.zeros_like(inp2)     # zero out precomputed text features
            else:
                inp2 = torch.ones_like(inp2)      # RoBERTa padding token ID = 1
            masks = torch.zeros_like(masks)       # zero attention mask in both cases

        inp1    = inp1.to(device)
        inp2    = inp2.to(device)
        masks   = masks.to(device)
        answers = answers.to(device)

        amp_ctx = autocast(device_type="cuda", dtype=AMP_DTYPE) if use_amp else nullcontext()
        with amp_ctx:
            logits, _ = model(inp1, inp2, masks)

        preds = logits.argmax(dim=1)
        vqa_scores = answers[torch.arange(answers.size(0), device=answers.device), preds]
        vqa_acc_sum += vqa_scores.sum().item()
        total += answers.size(0)

    return vqa_acc_sum / total * 100

# Use models + val_loader
models_dict = {"Symmetric": symmetric_model, "Asymmetric": asymmetric_model}

conditions = ["Full Data", "Image-Blind", "Text-Blind"]
ablation_results = {"Symmetric": {}, "Asymmetric": {}}

for model_name, model in models_dict.items():
    print(f"\nRunning ablation for {model_name}...")
    for cond in conditions:
        acc = evaluate_ablation(model, val_loader, cond)
        ablation_results[model_name][cond] = acc
        print(f"  {cond}: {acc:.2f}%")

# Plotting the results
x = np.arange(len(conditions))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 6))
sym_accs = [ablation_results["Symmetric"][cond] for cond in conditions]
asym_accs = [ablation_results["Asymmetric"][cond] for cond in conditions]

bars1 = ax.bar(x - width/2, sym_accs, width, label='Symmetric', color='lightblue')
bars2 = ax.bar(x + width/2, asym_accs, width, label='Asymmetric', color='steelblue')

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=9)

ax.set_ylabel('Accuracy (%)')
ax.set_title('Modality Ablation Test - Performance Degradation')
ax.set_xticks(x)
ax.set_xticklabels(conditions)
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES_DIR / "ablation_test.png", dpi=150, bbox_inches="tight")
plt.show()


# Task
Create a helper function `visualize_category_example` that searches the validation dataset for a question of a specified category and plots the attention heatmaps for both models. Use this function to find and visualize examples for the 'Yes/No', 'Count', 'Color', and 'Other' categories, adding appropriate text and code cells for each, and conclude with a brief summary of these visualizations.

## Define Category Visualization Helper

### Subtask:
Create a helper function to search for and visualize a specific question category.


In [ ]:
def visualize_category_example(category, dataset, models_dict, idx_to_answer):
    print(f"Searching for an example in category: {category}...")
    for i in range(len(dataset)):
        question = dataset.samples[i]["question"]
        if categorize_question(question) == category:
            image, input_ids, attention_mask, answer_target = dataset[i]
            gt_answer = idx_to_answer[answer_target.argmax().item()]

            img_t = image.unsqueeze(0).to(device)
            ids_t = input_ids.unsqueeze(0).to(device)
            mask_t = attention_mask.unsqueeze(0).to(device)

            # Decode tokens to reconstruct a readable question
            tokens = decode_tokens(input_ids)
            clean_tokens = [t.replace('Ġ', '') for t in tokens if t not in ['<pad>', '<s>', '</s>']]
            question_str = " ".join(clean_tokens).strip() + "?"

            print("\n" + "="*40)
            print(f"Category: {category}")
            print(f"Question: {question_str}")
            print(f"Target Answer: {gt_answer}")
            print("="*40)

            for model_name, model in models_dict.items():
                model.eval()
                with torch.no_grad():
                    logits, _ = model(img_t, ids_t, mask_t)
                pred_answer = idx_to_answer.get(logits.argmax(dim=1).item(), "???")
                print(f"{model_name} Guess: {pred_answer}")

            for model_name, model in models_dict.items():
                print(f"\n=== {model_name.upper()} MODEL ATTENTION ===")
                attn = get_attention_weights(model, image, input_ids, attention_mask)

                # Plot text -> image attention
                fig_img = plot_image_attention(attn["txt_to_img"], image, tokens, question_str)
                plt.show()

                # Plot image -> text attention
                fig_txt = plot_text_attention(attn["img_to_txt"], tokens, question_str)
                plt.show()

            return

    print(f"No example found for category: {category}")

## Visualize Yes/No Question

### Subtask:
Find and visualize an example for the 'Yes/No' question category.


In [ ]:
visualize_category_example("Yes/No", raw_val_ds, e2e_models_dict, idx_to_answer)

## Visualize Count Question

### Subtask:
Find and visualize an example for the 'Count' question category.

In [ ]:
visualize_category_example("Count", raw_val_ds, e2e_models_dict, idx_to_answer)

## Visualize Color Question

### Subtask:
Find and visualize an example for the 'Color' question category.

In [ ]:
visualize_category_example("Color", raw_val_ds, e2e_models_dict, idx_to_answer)

## Visualize Other Question

### Subtask:
Find and visualize an example for the 'Other' question category.

In [ ]:
visualize_category_example("Other", raw_val_ds, e2e_models_dict, idx_to_answer)

## Summary of Category Visualizations

Based on the attention heatmaps across different question categories ('Yes/No', 'Count', 'Color', 'Other'), we can observe the following:

1. **Asymmetric Model Focus**: The asymmetric model often demonstrates a more focused and interpretable attention mechanism. When asked about a specific object or its attribute (like 'color' or 'count'), the text-to-image attention effectively isolates the relevant regions of the image corresponding to the target words.
2. **Symmetric Model Limitations**: The symmetric model's attention maps tend to be more diffuse. Because it uses a single shared block for both directions, it struggles to decouple the distinct tasks of 'understanding the question' and 'locating the visual evidence'.
3. **Question-Specific Grounding**: In 'Count' and 'Color' questions, grounded visual evidence is crucial. The asymmetric model's ability to first process the text and then use it as a query to attend to the image allows it to better pinpoint the items to be counted or analyzed for color, leading to more accurate predictions.

In [ ]:
!mkdir -p /content/data/zip/
!mkdir -p /content/data/images/
!cp /content/drive/MyDrive/test2015.zip /content/data/zip/

In [ ]:
!unzip -q /content/data/zip/test2015.zip -d /content/data/images/

---
## 6. Test Set Predictions Export

Run inference on the VQA v2.0 **test2015** split (no annotations) and write predictions to JSON in the official VQA submission format: `[{"question_id": int, "answer": str}, ...]`.

Test images aren't in the precomputed feature HDF5, so we load raw JPEGs from `images/test2015/` and run them through the end-to-end models (`e2e_models_dict`, which works for both frozen and unfrozen runs).

In [ ]:
class VQATestDataset(Dataset):
    """VQA test split: questions + raw JPEGs from images/test2015/, no annotations."""

    def __init__(self, questions_file, images_dir, max_question_len=20,
                 transform=None, max_samples=None):
        self.images_dir = Path(images_dir) / "test2015"
        self.max_question_len = max_question_len
        self.transform = transform or get_image_transform("val")
        self.tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

        with open(questions_file) as f:
            self.samples = json.load(f)["questions"]
        if max_samples is not None:
            self.samples = self.samples[:max_samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        # test-dev2015 reuses the test2015 image pool, so the filename prefix is
        # always COCO_test2015_* regardless of which question split we loaded.
        img_path = self.images_dir / f"COCO_test2015_{sample['image_id']:012d}.jpg"
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        encoding = self.tokenizer(
            sample["question"],
            max_length=self.max_question_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return (
            image,
            encoding["input_ids"].squeeze(0),
            encoding["attention_mask"].squeeze(0),
            sample["question_id"],
        )


# Use the full test2015 split (~447K questions) — required for any VQA v2
# eval-server submission (test-dev or test-standard). For a quick local
# sanity check during iteration, swap to v2_OpenEnded_mscoco_test-dev2015_questions.json (~107K).
TEST_QUESTIONS_FILE = DATA_DIR / "questions" / "v2_OpenEnded_mscoco_test2015_questions.json"

test_ds = VQATestDataset(
    questions_file=TEST_QUESTIONS_FILE,
    images_dir=DATA_DIR / "images",
    max_question_len=MAX_QUESTION_LEN,
    transform=get_image_transform("val"),
    max_samples=None,  # always export the full test split, regardless of dev MAX_SAMPLES
)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)
print(f"Test: {len(test_ds):,} samples ({len(test_loader)} batches)")


@torch.no_grad()
def predict_test(model, loader, idx_to_answer):
    """Return [{question_id, answer}, ...] in official VQA submission format."""
    model.eval()
    use_amp = USE_AMP and device.type == "cuda"
    predictions = []

    for images, input_ids, attention_mask, question_ids in tqdm(loader, desc="predict"):
        images = images.to(device)
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        amp_ctx = autocast(device_type="cuda", dtype=AMP_DTYPE) if use_amp else nullcontext()
        with amp_ctx:
            logits, _ = model(images, input_ids, attention_mask)

        preds = logits.argmax(dim=1).cpu().tolist()
        for qid, p in zip(question_ids.tolist(), preds):
            predictions.append({"question_id": int(qid), "answer": normalize_answer(idx_to_answer[int(p)])})

    return predictions


PREDICTIONS_DIR = CHECKPOINT_DIR.parent / "predictions"
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

for name, model in [("asymmetric", e2e_models_dict["Asymmetric"]),
                    ("symmetric",  e2e_models_dict["Symmetric"])]:
    preds = predict_test(model, test_loader, idx_to_answer)
    out_path = PREDICTIONS_DIR / f"{RUN_TAG}_{name}_test_predictions.json"
    with open(out_path, "w") as f:
        json.dump(preds, f)
    print(f"Saved {len(preds):,} {name} predictions -> {out_path}")


In [ ]:
!mkdir -p /content/drive/MyDrive/unfrozen_results_ViT_CLIP_RoBERTa/

!cp -r /content/results/ /content/drive/MyDrive/unfrozen_results_ViT_CLIP_RoBERTa/